# Experiments Section 9.1 — Table 2b and Figure 1b: APE

This notebook reproduces the finite-shift APE experiment at δ=1 in Section 9.1. It uses the same nonlinear Gaussian DGP, two-fold cross-fitting, a common MLP outcome learner, and the three methods reported in Table 2b: Data-SMR, Time-SMR, and Riesz regression.

In [ ]:

from pathlib import Path
import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = Path.cwd()
for parent in [REPO, *REPO.parents]:
    if (parent / "src" / "genriesz" / "scorematchingriesz.py").exists():
        REPO = parent
        break
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import genriesz.scorematchingriesz as smr
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
print("device:", DEVICE)


In [ ]:
N_TRIALS = 200
N = 1000
N_FOLDS = 2
DELTA = 1.0
N_MC_TRUTH = 200000
HIDDEN_DIMS = (256, 256, 256)
OUTCOME_EPOCHS = 200
SCORE_STEPS = 4000
RATIO_STEPS = 4000
BATCH_SIZE = 256
INTEGRATION_STEPS = 200
DSM_SIGMA_MIN = 0.01
DSM_SIGMA_MAX = 1.0
SIGMA_EVAL = 0.01
CLIP_LOG_RATIO = 20.0
TABLE_TITLE = "Table 2b: APE performance metrics"
FIGURE_TITLE = "Figure 1b: APE estimation errors"

In [ ]:

def mu_function(x):
    x = np.asarray(x)
    return (
        1.0 + x[:, 0]
        + 0.1 * x[:, 0] ** 2
        + 2.0 * np.sin(x[:, 0])
        + x[:, 1]
        + x[:, 0] * x[:, 1]
        + x[:, 2] ** 2
        + x[:, 2] ** 3
    )


def partial_d_mu(x):
    x = np.asarray(x)
    return 1.0 + 0.2 * x[:, 0] + 2.0 * np.cos(x[:, 0]) + x[:, 1]


def sample_x(n, seed):
    rng = np.random.default_rng(seed)
    sigma = np.array([[1.0, 0.1, 0.1], [0.1, 1.0, 0.1], [0.1, 0.1, 1.0]])
    return rng.multivariate_normal(np.zeros(3), sigma, size=n).astype("float32")


def sample_observations(n, seed):
    rng = np.random.default_rng(seed)
    x = sample_x(n, seed)
    y = mu_function(x) + rng.normal(size=n)
    return x.astype("float32"), y.astype("float32")


def shift_x(x, delta):
    out = np.asarray(x, dtype="float32").copy()
    out[:, 0] += float(delta)
    return out


def true_ame(n_mc, seed=999):
    x = sample_x(n_mc, seed)
    return float(np.mean(partial_d_mu(x)))


def true_ape(delta, n_mc, seed=777):
    x = sample_x(n_mc, seed)
    return float(np.mean(mu_function(shift_x(x, delta)) - mu_function(shift_x(x, -delta))))


def summarize_trials(df, group_cols):
    return (
        df.groupby(group_cols)
        .agg(
            trials=("estimate", "count"),
            truth=("truth", "mean"),
            bias=("error", "mean"),
            mse=("error", lambda s: float(np.mean(np.square(s)))),
            coverage=("covered", "mean"),
            avg_se=("se", "mean"),
        )
        .reset_index()
    )


In [ ]:

TRUE_APE = true_ape(DELTA, N_MC_TRUTH)
METHODS = ["Data-SMR", "Time-SMR", "Riesz reg."]
print("True APE:", TRUE_APE)


def ratio_time_or_riesz(method, x_train, x_q_train, x_eval, seed):
    if method == "Time-SMR":
        model = smr.fit_time_smr_dre_infinity(
            x_q_train, x_train, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS,
            batch_size=BATCH_SIZE, seed=seed, device=DEVICE
        )
        log_r = smr.log_ratio_from_time_score(model, x_eval, steps=INTEGRATION_STEPS, normalize=True, x_p_for_norm=x_train, device=DEVICE)
        return np.exp(np.clip(log_r.detach().cpu().numpy().reshape(-1), -CLIP_LOG_RATIO, CLIP_LOG_RATIO))
    if method == "Riesz reg.":
        model = smr.fit_sq_riesz_ratio(
            x_q_train, x_train, hidden_dims=HIDDEN_DIMS, n_steps=RATIO_STEPS,
            batch_size=BATCH_SIZE, seed=seed, device=DEVICE
        )
        return smr.eval_ratio_sq(model, x_eval, normalize=True, x_p_for_norm=x_train, device=DEVICE).reshape(-1)
    raise ValueError(method)


def estimate_ape_trial(seed):
    x, y = sample_observations(N, seed)
    rows = []
    for method in METHODS:
        score_values = np.zeros(N)
        for train_idx, test_idx in smr.crossfit_splits(N, n_folds=N_FOLDS, seed=seed):
            x_train, y_train = x[train_idx], y[train_idx]
            x_test, y_test = x[test_idx], y[test_idx]
            outcome = smr.fit_outcome_net(x_train, y_train, hidden_dims=HIDDEN_DIMS, n_epochs=OUTCOME_EPOCHS, batch_size=BATCH_SIZE, seed=seed, device=DEVICE)
            gamma_hat = smr.predict_outcome(outcome, x_test, device=DEVICE).reshape(-1)
            m_gamma = (
                smr.predict_outcome(outcome, shift_x(x_test, DELTA), device=DEVICE).reshape(-1)
                - smr.predict_outcome(outcome, shift_x(x_test, -DELTA), device=DEVICE).reshape(-1)
            )
            if method == "Data-SMR":
                score_model = smr.fit_data_smr_score_dsm(x_train, hidden_dims=HIDDEN_DIMS, n_steps=SCORE_STEPS, batch_size=BATCH_SIZE, sigma_min=DSM_SIGMA_MIN, sigma_max=DSM_SIGMA_MAX, seed=seed, device=DEVICE)
                log_plus = smr.log_ratio_from_data_score_shift(score_model, x_test, DELTA, steps=INTEGRATION_STEPS, sigma_eval=SIGMA_EVAL, direction="+", normalize=True, x_p_for_norm=x_train, device=DEVICE).reshape(-1)
                log_minus = smr.log_ratio_from_data_score_shift(score_model, x_test, DELTA, steps=INTEGRATION_STEPS, sigma_eval=SIGMA_EVAL, direction="-", normalize=True, x_p_for_norm=x_train, device=DEVICE).reshape(-1)
                alpha_hat = np.exp(np.clip(log_plus, -CLIP_LOG_RATIO, CLIP_LOG_RATIO)) - np.exp(np.clip(log_minus, -CLIP_LOG_RATIO, CLIP_LOG_RATIO))
            else:
                r_plus = ratio_time_or_riesz(method, x_train, shift_x(x_train, DELTA), x_test, seed)
                r_minus = ratio_time_or_riesz(method, x_train, shift_x(x_train, -DELTA), x_test, seed + 17)
                alpha_hat = r_plus - r_minus
            score_values[test_idx] = m_gamma + alpha_hat * (y_test - gamma_hat)
        est = smr.wald_interval(score_values)
        rows.append({"method": method, "estimate": est.estimate, "se": est.se, "ci_low": est.ci_low, "ci_high": est.ci_high, "truth": TRUE_APE, "error": est.estimate - TRUE_APE, "covered": est.ci_low <= TRUE_APE <= est.ci_high})
    return rows

trial_rows = []
for trial in range(N_TRIALS):
    trial_rows.extend([{**row, "trial": trial} for row in estimate_ape_trial(RANDOM_SEED + trial)])
ape_results = pd.DataFrame(trial_rows)
print(TABLE_TITLE)
display(summarize_trials(ape_results, ["method"]))


In [ ]:

fig, ax = plt.subplots(figsize=(7, 4))
methods = list(ape_results["method"].unique())
ax.boxplot([ape_results.loc[ape_results["method"] == m, "error"] for m in methods], labels=methods, showfliers=False)
ax.axhline(0.0, linestyle="--")
ax.set_title(FIGURE_TITLE)
ax.set_ylabel("estimate minus truth")
fig.tight_layout()
plt.show()
